(1) Packages Installation

In [12]:
! uv pip install ipykernel langchain_community tiktoken langchain-openai langchainhub chromadb langchain bs4 langchain-text-splitters

Checked 9 packages in 103ms


(2) Setup Langsmith
- Better Observability
- Better analytics
- Metrices
- Logging

https://docs.smith.langchain.com/

In [1]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = 'https://apac.api.smith.langchain.com'
# os.environ["LANGCHAIN_API_KEY"] = "your-langchain-api-key"
assert "LANGCHAIN_API_KEY" in os.environ, "LANGCHAIN_API_KEY NOT SET"

(3) Setup LLM API key
- You can set it up in the notebook
- Or you can set it up in yoru `$HOME/.zshrc` | `$HOME/.profile`

In [2]:
# os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
assert "OPENAI_API_KEY" in os.environ, "OPEN_API_KEY NOT SET"

(4) Set up USER_AGENT for fetching documents from the web (optional)

In [3]:
os.environ["USER_AGENT"] = "indexmind-rag/1.0 (contact: pramitsingh0@gmail.com)"

# PART 1: Indexing
- Indexing is the process of locating a specific data point in a data set.
- For this process, first documents are split into chunks.
- Then, each chunk is vectorized and embedded into a vector space.
- The input question is also vectorized and embedded into a vector space.
- Indexing makes queries faster by reducing the number of chunks that need to be searched.

In [4]:
# documents
question = "What kind of pets do I like?"
document = "My favourite pet is a dog."

Count tokens considering ~4 characters per token

In [5]:
import tiktoken

def num_tokens_from_string(strings: list[str], encoding_name: str = "cl100k_base") -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = 0
    for s in strings:
        num_tokens += len(encoding.encode(s))
    return num_tokens

num_tokens_from_string([question])

8

Next we do embedding of the question and the documents. We use the OpenAI embeddings model.
- Embedding is the process of taking a text input and converting it into a numerical representation.
- Why numerical representation? For better comparison between similarity of question and related documents.

In [6]:
from langchain_openai import OpenAIEmbeddings

embd = OpenAIEmbeddings()
query_result = embd.embed_query(question)
documents_result = embd.embed_query(document)

Next we compare the query result and the document result.
[Cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity) is recommended (1 indicates identical, 0 indicates completely different) for OpenAI embeddings.
Usually we don't need to do this manually as chromadb does the comparison when used as an retriever.

In [7]:
import numpy as np

def consine_similarity(vec1, vec2) -> float:
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity: float = consine_similarity(query_result, documents_result) # we pass the two vectors
print("Cosine similariy: ", similarity)

Cosine similariy:  0.8688753215233779


Next we load documents, from different sources either from the web or from a local folder.
We use `WebBaseLoader` to loader to load documents from web

In [8]:
from langchain_core.documents import Document
from bs4.filter import SoupStrainer

from langchain_community.document_loaders import WebBaseLoader

# add documents here
document_links = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/"
]

loader = WebBaseLoader(
    web_paths=document_links,
    bs_kwargs=dict(
        parse_only=SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

blog_docs: list[Document] = loader.load()


/var/folders/zj/yk91t0ns0ls8vpqlppvsx7z80000gn/T/ipykernel_40725/1900164770.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


[Splitter](https://python.langchain.com/docs/modules/data_connection/document_transformers/recursive_text_splitter)

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

conver the documents to chunks of text, and then split each chunk into smaller chunks, recursively

Basically think of each chunk as a page you hand to someone to read, chunk size is roughly the number of chars in the page that is to be included in that specific chunk.
and chunk overlap is the next page repeats up to 50 or so character form the previous page, so an idea split at the boundary keeps the prev page context.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

splits: list[Document] = text_splitter.split_documents(blog_docs)

# Part 2: Retrieval
[Vectorstores](https://python.langchain.com/docs/integrations/vectorstores/)

A vectorstore is a database of embeddings that performs similarity search over the embeddings. It is a way to store and retrieve embeddings efficiently.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore: Chroma = Chroma.from_documents(documents=splits,
                                    embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()
# retriever: VectorStoreRetriever = vectorstore.as_retriever(search_kwargs={"k": 1}) # this is kind of knn, like k nearest neighbours meaning it will return the 1 closest vectors to the query
# docs: list[Document] = retriever.invoke(question)
# len(docs)

1

# Part 3: Generation
- This part introduces the generation of answers to the query from the retrieved documents
- The context and the question is formatted as a dictionary
- A prompt template is used to generate the prompt with the context and the question injected
- the prompt is then passed to the LLM to generate the answer

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [15]:
# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [16]:
# chain
chain = prompt | llm

In [18]:
# Run
chain.invoke({"context": blog_docs, "question": "What is task decomposition?"})

AIMessage(content='Task decomposition is the process by which an agent breaks down large tasks into smaller, manageable subgoals. This enables the agent to handle complex tasks more efficiently by transforming big tasks into multiple simpler steps. Task decomposition can be achieved through various methods, such as prompting the LLM with specific questions about the steps needed to accomplish a task, using task-specific instructions, or incorporating human inputs.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 9988, 'total_tokens': 10065, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint'

In [ ]:
from langchain_classic import hub
prompt_hub_rag = hub.pull("rlm/rag-prompt")

In [ ]:
prompt_hub_rag

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, question: RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is task decomposition?")

SyntaxError: invalid syntax (685443838.py, line 8)